# Merge Sort Lab

```{contents}
:local:
:depth: 2
```

A finance team sorts invoices by amount while retaining the arrival order of equal amounts. Build correctness checks before comparing counts. Every code cell includes its own setup and implementation.

Download {download}`the complete stable merge-sort source and reference checks <../../demos/21/StableSort.cs>` for a console project.


## Trace Merge Fronts

```{index} Trace Merge Fronts
```

Track consumed indexes i and j and output index k. Equal keys take the left item first. Predict which side runs out, then trace both the comparison loop and the tail copies.

In [ ]:
using System;
using System.Linq;

int[] left={1,4,4},right={2,4,9};
int i=0,j=0,k=0;int[] result=new int[left.Length+right.Length];
while(i<left.Length && j<right.Length)
{
 Console.WriteLine($"i={i}, j={j}, k={k}; fronts={left[i]}/{right[j]}");
 result[k++]=left[i]<=right[j]?left[i++]:right[j++];
}
while(i<left.Length)result[k++]=left[i++];
while(j<right.Length)result[k++]=right[j++];
Console.WriteLine(string.Join(", ",result));

The invariant k=i+j and sorted consumed-prefix merge explains why no value disappears. The measure (left.Length−i)+(right.Length−j) decreases after every copy. Stability is relative to a comparison key; equal integer values cannot display record order, so use labeled invoices next.

## Sort Ranges with One Buffer

```{index} Sort Ranges with One Buffer
```

The public method clones the input once and allocates one reusable buffer. SortRange(start,end) sorts a half-open range. Sibling calls finish sequentially, then merge their sorted ranges into the matching buffer range and copy back. Counts reset at each Sort call.

In [ ]:
using System;
using System.Linq;

int[] values={7,2,6,3,1,5,4};
var counts=new StableSort.Counts();
int[] sorted=StableSort.Sort(values,(a,b)=>a.CompareTo(b),counts);
Console.WriteLine($"input: {string.Join(", ",values)}");
Console.WriteLine($"sorted: {string.Join(", ",sorted)}");
Console.WriteLine($"comparisons={counts.Comparisons}, writes={counts.Writes}, calls={counts.Calls}, peak={counts.PeakFrames}");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


Assume each recursive call correctly sorts its smaller range and preserves its items. The merge invariant establishes the same promise for the parent. Empty/singleton ranges return directly. Odd lengths are fully covered. The copied output and buffer use Θ(n) storage, and the stack adds Θ(log n); total auxiliary space is Θ(n). Across all levels there are Θ(n log n) writes, but cumulative writes are not peak memory. The original slice version also allocates across levels; separate total allocation volume from peak live storage.

## Verify Stability with Labeled Invoices

```{index} Verify Stability with Labeled Invoices
```

A stable sort preserves arrival order among equal amounts. Compare only the amount, and track identity separately. A consistent comparison must define an ordering; subtracting integer keys can overflow, so use CompareTo.

In [ ]:
using System;
using System.Linq;

var invoices=new[]{("A",200),("B",100),("C",200),("D",100),("E",200)};
var counts=new StableSort.Counts();
var sorted=StableSort.Sort(invoices,(a,b)=>a.Item2.CompareTo(b.Item2),counts);
Console.WriteLine(string.Join(", ",sorted.Select(x=>$"{x.Item1}:{x.Item2}")));
if(!sorted.Select(x=>x.Item1).SequenceEqual(new[]{"B","D","A","C","E"}))throw new Exception("Stability failed.");
Console.WriteLine("stable invoice order passed");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


Changing the merge comparison from ≤0 to <0 would let equal right items overtake equal left items. Stability matters when reports preserve earlier arrival order or apply successive sorts. It is a property of the algorithm and comparison policy, not merely of sorted numeric output.

## Check Order, Preservation, and Copying

```{index} Check Order, Preservation, and Copying
```

Sortedness alone is insufficient: returning an empty array is sorted but loses every invoice. Compare against an independent sorted reference, check input preservation and array identity, and include empty, singleton, odd, duplicate, reverse, negative, and extreme cases.

In [ ]:
using System;
using System.Linq;

int[][] cases={Array.Empty<int>(),new[]{4},new[]{3,1,2},new[]{2,2,1,2},new[]{5,4,3,2,1},new[]{int.MaxValue,0,int.MinValue}};
foreach(int[] values in cases)
{
 int[] before=(int[])values.Clone();var counts=new StableSort.Counts();
 int[] actual=StableSort.Sort(values,(a,b)=>a.CompareTo(b),counts);
 if(!actual.SequenceEqual(before.OrderBy(x=>x)))throw new Exception("Order/preservation mismatch.");
 if(!values.SequenceEqual(before)||ReferenceEquals(values,actual))throw new Exception("Copy contract failed.");
 Console.WriteLine($"[{string.Join(",",values)}] -> [{string.Join(",",actual)}] verified");
}
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


## Compare Measured Operations

```{index} Compare Measured Operations
```

Count actual key comparisons and element writes. The asymptotic expression n log₂ n is a growth model, not an exact comparison count. Different orders can exhaust one merge side earlier. Buffer copy-back contributes writes even after comparisons stop.

In [ ]:
using System;
using System.Linq;

foreach(int n in new[]{4,8,16,32})
{
 var counts=new StableSort.Counts();int[] values=Enumerable.Range(0,n).Reverse().ToArray();
 StableSort.Sort(values,(a,b)=>a.CompareTo(b),counts);
 Console.WriteLine($"n={n}, comparisons={counts.Comparisons}, writes={counts.Writes}, peak={counts.PeakFrames}, nlogn={n*Math.Log2(n):F0}");
}
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


For powers of two, this implementation performs exactly 2n log₂ n counted writes: one buffer write and one copy-back write per item per internal level. Clone and buffer allocation are setup and are excluded from that counter, but included in the algorithm’s overall cost. Recursion has 2n−1 calls for nonempty input. Time remains Θ(n log n); these are deterministic operation counts, not timings.

## Exercise

```{index} Exercise
```

Sort labeled invoices by amount in descending order while preserving arrival order for ties. Test int.MinValue and int.MaxValue safely. Reverse the comparison operands; do not subtract keys.

In [ ]:
using System;
using System.Linq;

var invoices=new[]{("A",3),("B",int.MinValue),("C",3),("D",int.MaxValue)};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
var invoices=new[]{("A",3),("B",int.MinValue),("C",3),("D",int.MaxValue)};
var sorted=StableSort.Sort(invoices,(a,b)=>b.Item2.CompareTo(a.Item2),new StableSort.Counts());
Console.WriteLine(string.Join(", ",sorted.Select(x=>x.Item1)));
if(!sorted.Select(x=>x.Item1).SequenceEqual(new[]{"D","A","C","B"}))throw new Exception("Descending stability failed.");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


D, A, C, B


```{rubric} Footnotes
```
[^1]: This implementation sorts a copy using one reusable merge buffer. It does not claim in-place sorting or constant auxiliary memory.
[^2]: Stability depends on a consistent key comparison and left-first handling of equal keys at every merge.